In [1]:
!apt-get update
!apt-get install -y nmap

!pip install streamlit plotly pandas requests python-nmap -q

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,856 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.9 k

In [2]:
TARGETS = [
    "testasp.vulnweb.com",
    "testphp.vulnweb.com",
    "zero.webappsecurity.com"
]

SCAN_OUTPUT_DIR = "scan_xml"

In [3]:
import os
import subprocess

os.makedirs(SCAN_OUTPUT_DIR, exist_ok=True)

for target in TARGETS:

    output_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"

    cmd = [
        "nmap",
        "-sV",
        "-oX",
        output_file,
        target
    ]

    subprocess.run(cmd)

print("Scanning complete")

Scanning complete


In [4]:
import pandas as pd
import xml.etree.ElementTree as ET

results = []

for target in TARGETS:

    xml_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"

    tree = ET.parse(xml_file)
    root = tree.getroot()

    for host in root.findall("host"):

        ip = host.find("address").get("addr")

        ports = host.find("ports")

        if ports is None:
            continue

        for port in ports.findall("port"):

            port_id = port.get("portid")

            service = port.find("service")

            if service is not None:
                service_name = service.get("name")
            else:
                service_name = "unknown"

            results.append({
                "ip": ip,
                "port": port_id,
                "service": service_name
            })

df = pd.DataFrame(results)

df.head()

,ip,port,service
0,44.238.29.244,80,http
1,54.82.22.214,80,http
2,54.82.22.214,443,https
3,54.82.22.214,8080,http


In [ ]:
import requests

VT_API_KEY = "Your Virus Total API Key"

In [ ]:
SENDER_EMAIL = "sender@gmail.com"
APP_PASSWORD = "xxxxxxxxxxxxxxxxxxxx"
RECEIVER_EMAIL = "receiver@gmail.com"

In [7]:
def check_virustotal(ip):

    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"

    headers = {
        "x-apikey": VT_API_KEY
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        return "Unknown"

    data = response.json()

    malicious = data["data"]["attributes"]["last_analysis_stats"]["malicious"]

    if malicious > 5:
        return "High"
    elif malicious > 0:
        return "Medium"
    else:
        return "Low"

In [8]:
HIGH_RISK_SERVICES = {
    "ftp":3,
    "telnet":4,
    "ssh":1
}

def calculate_risk(row):

    score = 1

    if row["service"] in HIGH_RISK_SERVICES:
        score += HIGH_RISK_SERVICES[row["service"]]

    return score

In [9]:
df["risk_score"] = df.apply(calculate_risk, axis=1)

df["vt_reputation"] = df["ip"].apply(check_virustotal)

In [10]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

def send_email_alert(df,target):

    high_risk = df[df["severity"].isin(["High","Critical"])]

    if high_risk.empty:
        print("No high risk vulnerabilities — email not sent")
        return

    msg = MIMEMultipart()

    msg["From"] = SENDER_EMAIL
    msg["To"] = RECEIVER_EMAIL
    msg["Subject"] = f"[SECURITY ALERT] High risk vulnerabilities detected on {target}"

    html = f"""
    <h2>CyberScan Pro Alert</h2>

    Target: {target}<br>
    Scan Time: {datetime.now()}<br>

    <h3>High Risk Findings</h3>

    {high_risk.to_html(index=False)}

    <hr>
    <i>Automatically generated by CyberScan Pro.</i>
    """

    msg.attach(MIMEText(html,"html"))

    server = smtplib.SMTP("smtp.gmail.com",587)
    server.starttls()
    server.login(SENDER_EMAIL,APP_PASSWORD)
    server.sendmail(SENDER_EMAIL,RECEIVER_EMAIL,msg.as_string())
    server.quit()

    print("Alert email sent")

In [11]:
# Define a function to assign severity based on risk_score and vt_reputation
def assign_severity(row):
    if row["vt_reputation"] == "High" or row["risk_score"] >= 4:
        return "High"
    elif row["vt_reputation"] == "Medium" or row["risk_score"] >= 2:
        return "Medium"
    else:
        return "Low"

# Apply the function to create the 'severity' column
df["severity"] = df.apply(assign_severity, axis=1)

df.to_csv("scan_results.csv", index=False)
send_email_alert(df, "testphp.vulnweb.com")
print("scan_results.csv saved")

No high risk vulnerabilities — email not sent
scan_results.csv saved


In [29]:
%%writefile dashboard.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import subprocess
import xml.etree.ElementTree as ET
import requests
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

# =============================
# Global Variables and Functions
# =============================

TARGETS = [
    "testasp.vulnweb.com",
    "testphp.vulnweb.com",
    "zero.webappsecurity.com"
]

SCAN_OUTPUT_DIR = "scan_xml"

VT_API_KEY = "d3e1643b7d601a5f6c2cc35111b9af18a8c9763a3454b6bf5bb867f1bb2a1737" # Your actual VirusTotal API Key
SENDER_EMAIL = "yaswanthanbrcs225@gmail.com" # Your sender email
APP_PASSWORD = "drhsoqrqfqomqvuz" # Your app password
RECEIVER_EMAIL = "yashwanthan97@gmail.com" # Your receiver email

HIGH_RISK_SERVICES = {
    "ftp":3,
    "telnet":4,
    "ssh":1
}

def check_virustotal(ip):

    url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"

    headers = {
        "x-apikey": VT_API_KEY
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        return "Unknown"

    data = response.json()

    malicious = data["data"]["attributes"]["last_analysis_stats"]["malicious"]

    if malicious > 5:
        return "High"
    elif malicious > 0:
        return "Medium"
    else:
        return "Low"

def calculate_risk(row):

    score = 1

    if row["service"] in HIGH_RISK_SERVICES:
        score += HIGH_RISK_SERVICES[row["service"]]

    return score

def assign_severity(row):
    if row["vt_reputation"] == "High" or row["risk_score"] >= 4:
        return "High"
    elif row["vt_reputation"] == "Medium" or row["risk_score"] >= 2:
        return "Medium"
    else:
        return "Low"

def send_email_alert(df,target):

    # Reverted to include only 'High' and 'Critical' severity for alerts
    high_risk = df[df["severity"].isin(["High","Critical"])]

    if high_risk.empty:
        return "No high risk vulnerabilities — email not sent"

    msg = MIMEMultipart()

    msg["From"] = SENDER_EMAIL
    msg["To"] = RECEIVER_EMAIL
    msg["Subject"] = f"[SECURITY ALERT] High risk vulnerabilities detected on {target}"

    html = f"""
    <h2>CyberScan Pro Alert</h2>

    Target: {target}<br>
    Scan Time: {datetime.now()}<br>

    <h3>High Risk Findings</h3>

    {high_risk.to_html(index=False)}

    <hr>
    <i>Automatically generated by CyberScan Pro.</i>
    """

    msg.attach(MIMEText(html,"html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com",587)
        server.starttls()
        server.login(SENDER_EMAIL,APP_PASSWORD)
        server.sendmail(SENDER_EMAIL,RECEIVER_EMAIL,msg.as_string())
        server.quit()
        return "Alert email sent successfully."
    except Exception as e:
        return f"Failed to send email alert: {e}"

def run_full_scan_logic():
    os.makedirs(SCAN_OUTPUT_DIR, exist_ok=True)
    results = []

    with st.status("Running full scan...", expanded=True) as status_container:
        status_container.update(label="Initializing scan... (0%)", state="running")

        # Nmap Scanning
        status_container.update(label="Running Nmap scan... (10%)", state="running")
        for i, target in enumerate(TARGETS):
            output_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"
            cmd = [
                "nmap",
                "-sV",
                "-oX",
                output_file,
                target
            ]
            subprocess.run(cmd, capture_output=True, text=True)
            progress_percentage = 10 + int((i+1)/len(TARGETS) * 40)
            status_container.update(label=f"Scanning {target} with Nmap... ({progress_percentage}%)", state="running")
        status_container.write("Nmap scanning complete.")

        # Parsing results and VirusTotal check
        status_container.update(label="Parsing scan results and checking VirusTotal... (50%)", state="running")
        for i, target in enumerate(TARGETS):
            xml_file = f"{SCAN_OUTPUT_DIR}/{target}.xml"
            if not os.path.exists(xml_file):
                status_container.warning(f"Nmap XML file not found for {target}. Skipping VirusTotal check.")
                continue

            tree = ET.parse(xml_file)
            root = tree.getroot()

            for host in root.findall("host"):
                ip_element = host.find("address")
                if ip_element is None: # Sometimes nmap output doesn't have an address if target is down/unreachable
                    continue
                ip = ip_element.get("addr")

                ports = host.find("ports")

                if ports is None:
                    continue

                for port in ports.findall("port"):
                    port_id = port.get("portid")
                    service = port.find("service")
                    service_name = service.get("name") if service is not None else "unknown"

                    results.append({
                        "ip": ip,
                        "port": port_id,
                        "service": service_name
                    })
            progress_percentage = 50 + int((i+1)/len(TARGETS) * 40)
            status_container.update(label=f"Processing results for {target}... ({progress_percentage}%)", state="running")

        df_scan = pd.DataFrame(results)

        if not df_scan.empty:
            df_scan["risk_score"] = df_scan.apply(calculate_risk, axis=1)
            df_scan["vt_reputation"] = df_scan["ip"].apply(check_virustotal)
            df_scan["severity"] = df_scan.apply(assign_severity, axis=1)

            # Inject mock high vulnerability if none found for demonstration
            if df_scan[df_scan["severity"].isin(["High", "Critical"])].empty:
                mock_high_vuln = pd.DataFrame({
                    "ip": ["1.1.1.1"],
                    "port": [1337],
                    "service": ["mock_service"],
                    "risk_score": [10],
                    "vt_reputation": ["High"],
                    "severity": ["High"]
                })
                df_scan = pd.concat([df_scan, mock_high_vuln], ignore_index=True)
                status_container.info("Injected mock high vulnerability for demonstration purposes.")

            df_scan.to_csv("scan_results.csv", index=False)
            status_container.success("Scan results processed and saved.")
            status_container.update(label="Sending email alerts... (95%)", state="running")

            # Send email alert for the entire scan results if any high risk found
            # This ensures the mock vulnerability also triggers an email.
            high_risk_entries = df_scan[df_scan["severity"].isin(["High","Critical"])]
            if not high_risk_entries.empty:
                # Use a generic target name for the overall alert
                email_status_message = send_email_alert(df_scan, "Overall Scan Results")
                status_container.write(email_status_message)
            else:
                status_container.info("No high risk vulnerabilities found across all targets — email not sent")
        else:
            status_container.warning("No open ports or services found during scan.")
        status_container.update(label="Scan complete.", state="complete", expanded=False)
    st.cache_data.clear() # Clear cache to force reload data after scan

def load_scan_results():
    try:
        df = pd.read_csv("scan_results.csv")
        return df
    except FileNotFoundError:
        st.info("No scan run yet. Showing sample data. Please run a scan to get real results.")
        return pd.DataFrame({
            "ip":["192.168.1.1","192.168.1.1","192.168.1.2","192.168.1.2","192.168.1.3"],
            "port":[22,80,21,443,23],
            "service":["ssh","http","ftp","https","telnet"],
            "vt_reputation":["Low","Low","Medium","Medium","High"],
            "risk_score":[1,1,6,6,10],
            "severity":["Low","Low","High","High","High"]
        })

# =============================
# Streamlit App Layout
# =============================

st.set_page_config(
    page_title="CyberScan Pro",
    layout="wide"
)

st.title("🛡️ CyberScan Pro: Network Security Dashboard")
st.markdown("A powerful tool for network reconnaissance, threat intelligence, and vulnerability assessment.")
st.divider()

# Sidebar
st.sidebar.title("⚙️ Controls & Status")
st.sidebar.header("Application Status")

if VT_API_KEY and VT_API_KEY != "YOUR_VIRUSTOTAL_API_KEY":
    st.sidebar.success("✅ VirusTotal API Key configured")
else:
    st.sidebar.error("❌ VirusTotal API Key NOT configured")

if SENDER_EMAIL and APP_PASSWORD and RECEIVER_EMAIL:
    st.sidebar.success("📧 Email alert system configured")
else:
    st.sidebar.warning("⚠️ Email alert system NOT fully configured")

with st.sidebar.expander("🎯 Current Scan Targets", expanded=False):
    for target in TARGETS:
        st.write(f"- {target}")
st.sidebar.divider()

st.sidebar.header("🚀 Scan Actions")

if st.sidebar.button("🚀 Run New Full Scan", help="Execute a new Nmap scan and process results."):
    run_full_scan_logic()

if st.sidebar.button("🔄 Refresh Dashboard Data", help="Reload data from the last scan without running a new one."):
    st.cache_data.clear() # Clear cache to force reload data
    st.rerun()
st.sidebar.divider()

df = load_scan_results()

# Key Metrics
st.subheader("📊 Overview")
col1,col2,col3,col4,col5 = st.columns(5)

col1.metric("Total Hosts Found", df["ip"].nunique() if not df.empty else 0)
col2.metric("Open Ports Detected", len(df) if not df.empty else 0)
col3.metric("Unique Services", df["service"].nunique() if not df.empty else 0)
col4.metric("Highest Risk Score", df["risk_score"].max() if not df.empty else 0)
high_risk_count = len(df[df["severity"]=="High"]) if not df.empty else 0
col5.metric("High Risk Entries", high_risk_count, delta=f"Up from last scan" if high_risk_count > 0 else None)

st.divider()

# Navigation Tabs
tab1,tab2,tab3,tab4 = st.tabs([
"📄 Scan Data",
"📈 Visualizations",
"🚨 Threat Intelligence",
"💾 Export Data"
])

with tab1:
    st.header("Detailed Scan Results")
    if not df.empty:
        col_filter1, col_filter2 = st.columns(2)
        with col_filter1:
            ip_filter = st.selectbox("Filter by IP Address", ["All"] + list(df["ip"].unique()), key="ip_filter")
        with col_filter2:
            service_filter = st.selectbox("Filter by Service Name", ["All"] + list(df["service"].unique()), key="service_filter")

        filtered = df.copy()
        if ip_filter != "All":
            filtered = filtered[filtered["ip"]==ip_filter]
        if service_filter != "All":
            filtered = filtered[filtered["service"]==service_filter]

        st.dataframe(filtered, use_container_width=True, hide_index=True)

        st.markdown("### 📌 Host Summary")
        host_summary = df.groupby("ip").agg(
            open_ports=("port","count"),
            max_risk=("risk_score","max"),
            malicious_reputation_count=('vt_reputation', lambda x: (x == 'High').sum() + (x == 'Medium').sum()),
            services=("service",lambda x:", ".join(x.unique()))
        ).reset_index()
        st.dataframe(host_summary, use_container_width=True, hide_index=True)
    else:
        st.info("No scan data available. Run a full scan to populate results.")

with tab2:
    st.header("Data Visualizations")
    if not df.empty:
        col1,col2 = st.columns(2)
        with col1:
            ports_chart = px.bar(
                df.groupby("ip")["port"].count().reset_index(),
                x="ip",
                y="port",
                title="<b>Open Ports per Host</b>",
                labels={
                    "ip": "Host IP Address",
                    "port": "Number of Open Ports"
                },
                color_discrete_sequence=px.colors.qualitative.Plotly
            )
            st.plotly_chart(ports_chart, use_container_width=True)

        with col2:
            risk_chart = px.bar(
                df.groupby("ip")["risk_score"].sum().reset_index(),
                x="ip",
                y="risk_score",
                title="<b>Total Risk Score per Host</b>",
                labels={
                    "ip": "Host IP Address",
                    "risk_score": "Aggregated Risk Score"
                },
                color="risk_score",
                color_continuous_scale="Reds"
            )
            st.plotly_chart(risk_chart, use_container_width=True)

        col3,col4 = st.columns(2)
        with col3:
            service_chart = px.bar(
                df["service"].value_counts().reset_index(name='count', inplace=False),
                x="count",
                y="service",
                orientation="h",
                title="<b>Services Exposed Across All Hosts</b>",
                labels={
                    "count": "Number of Occurrences",
                    "service": "Service Name"
                },
                color_discrete_sequence=px.colors.qualitative.Pastel
            )
            st.plotly_chart(service_chart, use_container_width=True)

        with col4:
            severity_chart = px.pie(
                df,
                names="severity",
                title="<b>Severity Distribution of Findings</b>",
                hole=0.4,
                color="severity",
                color_discrete_map={'High':'red', 'Medium':'orange', 'Low':'green'}
            )
            st.plotly_chart(severity_chart, use_container_width=True)
    else:
        st.info("No scan data to display charts. Run a full scan.")

with tab3:
    st.header("🚨 Threat Intelligence Summary")
    if not df.empty:
        high_risk = df[df["severity"]=="High"]
        if len(high_risk)>0:
            st.error(
                f"⚠️ **CRITICAL ALERT:** {len(high_risk)} high-risk entries across "
                f"{high_risk['ip'].nunique()} host(s). Immediate action required."
            )
            with st.expander("🔴 Show High Risk Entries Details", expanded=True):
                st.dataframe(high_risk, use_container_width=True, hide_index=True)
        else:
            st.success("✅ No high risk entries detected. Your network appears secure.")
    else:
        st.info("No scan data available for threat intelligence. Run a full scan.")

with tab4:
    st.header("💾 Export Scan Results")
    if not df.empty:
        st.write("Download the full scan results as a CSV file.")
        csv = df.to_csv(index=False).encode("utf-8")
        st.download_button(
            label="Download CSV",
            data=csv,
            file_name="cyber_scan_results.csv",
            mime="text/csv",
            help="Click to download the current scan results as a CSV file."
        )
    else:
        st.info("No scan data to export. Please run a scan first.")


Overwriting dashboard.py


In [30]:
import subprocess
import threading

def run_streamlit():

    subprocess.run([
        "streamlit",
        "run",
        "dashboard.py",
        "--server.port","8501",
        "--server.headless","true"
    ])

threading.Thread(target=run_streamlit).start()

In [31]:
!nohup streamlit run dashboard.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

In [32]:
!cloudflared tunnel --url http://localhost:8501

2026-03-16T14:33:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-16T14:33:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-16T14:33:04Z INF +--------------------------------------------------------------------------------------------+
2026-03-16T14:33:04Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-16T14:33:04Z INF |  https://certified-lit-skins-topics.trycloudflare.com 

In [ ]:
import pandas as pd

# Create a dummy DataFrame with 'High' severity to test email alert
dummy_high_risk_df = pd.DataFrame({
    "ip": ["1.2.3.4"],
    "port": [22],
    "service": ["ssh"],
    "risk_score": [4],
    "vt_reputation": ["High"],
    "severity": ["High"]
})

print("Attempting to send email alert for dummy high-risk vulnerabilities...")
send_email_alert(dummy_high_risk_df, "dummy_test_target.com")
print("Verification complete.")

Attempting to send email alert for dummy high-risk vulnerabilities...
Alert email sent
Verification complete.
